# SET OS · Camera Coach v2 records run (M02)

This notebook is orchestration only. It uploads one deterministic **code bundle**, one **data bundle** of typed
`camera-training-record-v2` records, runs the fail-closed `preflight_records_run.py`, and then calls the single
training CLI that already exists in the repository:

```
python3 -m ml.camera_coach.train --config <runtime config> --run-dir <run dir>
```

No trainer, loss, loader or resume logic is copied into this notebook. The notebook never mounts Google Drive and
never performs an OAuth flow; it only asks the browser for the two files you already built on the workstation.

**Honest boundary.** Admitted records currently number **0** (Packet A is pending in
`docs/aegis/work/2026-09-03-gpt-5-6-pro-guidance/evidence-release/OWNER-PACKET.md`). The shipped profile is therefore
`non_admitted_research`: Run all performs a real, tiny, non-admitted dry run that proves the CLI, preflight, device,
checkpoint and resume path. It is **not** a quality or release model. A `full_fit` profile is refused by preflight
until Packet A provides admitted records.


In [ ]:
# ---- ONE PARAMETER BLOCK: fill these in and run the whole notebook ----
RUN_ID = "setos-records-20260913-01"

# The two archives produced by tools/dataset/package_camera_colab.py on the workstation.
CODE_BUNDLE_ZIP = "/content/SET_OS_Camera_Coach_Records.zip"
EXPECTED_CODE_BUNDLE_SHA256 = ""  # paste the sha256 printed by the packager (never the archive's own guess)
DATA_BUNDLE_ZIP = "/content/SET_OS_Camera_Coach_Records_DATA.zip"
EXPECTED_DATA_BUNDLE_SHA256 = ""  # paste the sha256 printed by the data packager

# Bundled frozen config template; the notebook only rebinds device/output_root/resume_from/runtime versions.
CONFIG_RELATIVE = "ml/camera_coach/configs/production_records_colab.json"

EXECUTION_PROFILE = "non_admitted_research"  # or "full_fit" once Packet A admits records
REQUIRE_CUDA = True                          # full fit must prove CUDA tensors; set False only for a CPU smoke
ALLOW_CPU_SMOKE = True                       # allows the non-admitted dry run on a CPU runtime when REQUIRE_CUDA is False
RESUME = True                                # resume the same RUN_ID if durable checkpoint semantics still match

# Local runtime disk (fast, ephemeral) and durable storage (survives a runtime restart).
RUNTIME_ROOT = f"/content/setos_camera_records_runtime/{RUN_ID}"
# Point this at an already-mounted persistent path if you want disconnect recovery. Do not mount Drive here.
DURABLE_DIR = "/content/setos_camera_records/durable"
# ---------------------------------------------------------------------


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

import torch

RUNTIME_ROOT = Path(RUNTIME_ROOT)
DURABLE_DIR = Path(DURABLE_DIR)
RUN_DIR_DURABLE = DURABLE_DIR / "runs" / RUN_ID
RUN_DIR_RUNTIME = RUNTIME_ROOT / "runs" / RUN_ID
CODE_ROOT = RUNTIME_ROOT / "code"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR_DURABLE.mkdir(parents=True, exist_ok=True)
for label, value in (("code", EXPECTED_CODE_BUNDLE_SHA256), ("data", EXPECTED_DATA_BUNDLE_SHA256)):
    if len(value) != 64 or any(char not in "0123456789abcdef" for char in value):
        raise RuntimeError(f"paste the lowercase {label} bundle sha256 into the parameter block before running")
print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "durable_dir": str(DURABLE_DIR),
    "durable_is_persistent_hint": not str(DURABLE_DIR).startswith("/content/setos_camera_records/"),
})


In [ ]:
# One browser upload of each archive; no OAuth and no Drive mount.
from google.colab import files

for path, label in ((Path(CODE_BUNDLE_ZIP), "code"), (Path(DATA_BUNDLE_ZIP), "data")):
    if not path.is_file():
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError(f"expected exactly one {label} archive upload, got {len(uploaded)}")
        name, payload = next(iter(uploaded.items()))
        if not name.lower().endswith(".zip"):
            raise RuntimeError(f"uploaded {label} artifact must be a .zip")
        path.write_bytes(payload)
    print(label, path, path.stat().st_size, "bytes")


In [ ]:
# Trust-anchor the code bundle and safely publish it onto the local runtime
# disk. preflight re-validates the archive policy and every extracted byte.
from pathlib import PurePosixPath
import stat
import zipfile

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

actual_code_sha256 = sha256_file(Path(CODE_BUNDLE_ZIP))
if actual_code_sha256 != EXPECTED_CODE_BUNDLE_SHA256:
    raise RuntimeError(f"code bundle trust anchor mismatch: {actual_code_sha256}")
if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
CODE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(Path(CODE_BUNDLE_ZIP)) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise RuntimeError("code bundle contains duplicate member names")
    for info in archive.infolist():
        member = PurePosixPath(info.filename)
        if member.is_absolute() or any(part in {"", ".", ".."} for part in member.parts):
            raise RuntimeError(f"unsafe code bundle member: {info.filename!r}")
        mode = info.external_attr >> 16
        if stat.S_ISLNK(mode):
            raise RuntimeError(f"code bundle member is a symlink: {info.filename!r}")
        if info.file_size > 4 * 1024 * 1024:
            raise RuntimeError(f"code bundle member exceeds the ceiling: {info.filename!r}")
        target = CODE_ROOT.joinpath(*member.parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(info) as source, target.open("wb") as sink:
            shutil.copyfileobj(source, sink)
print("code root:", CODE_ROOT, "members:", len(names), "sha256:", actual_code_sha256)


In [ ]:
# Derive the runtime config from the bundled frozen template. Only the
# preflight-allowed fields are rebound: device, output_root, resume_from and the
# interpreter/Torch versions. Training semantics are copied verbatim.
import tempfile
import zipfile

runtime_config_path = RUNTIME_ROOT / "run-config.json"
with zipfile.ZipFile(Path(CODE_BUNDLE_ZIP)) as archive:
    template = json.loads(archive.read(CONFIG_RELATIVE))
if template.get("runtime_profile") != "colab_installed_runtime":
    raise RuntimeError("the bundled config is not the colab_installed_runtime profile")

resume_from = None
if RESUME:
    durable_checkpoint = RUN_DIR_DURABLE / "seed-11" / "checkpoint.pt"
    if durable_checkpoint.is_file():
        resume_from = str(durable_checkpoint)
        print("resume checkpoint found:", resume_from)
    else:
        print("no durable checkpoint yet; this is a fresh run")

template["device"] = "cuda" if torch.cuda.is_available() else "cpu"
template["runtime"]["python_version"] = sys.version.split()[0]
template["runtime"]["torch_version"] = torch.__version__
template["output_root"] = str(RUNTIME_ROOT / "runs")
template["resume_from"] = resume_from
runtime_config_path.write_text(json.dumps(template, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("runtime config:", runtime_config_path)


In [ ]:
# Fail-closed preflight. A non-zero exit stops the notebook before any fit.
preflight_receipt_path = DURABLE_DIR / "preflight-receipt.json"
preflight_command = [
    sys.executable,
    str(CODE_ROOT / "ml" / "camera_coach" / "preflight_records_run.py"),
    "--code-bundle", str(Path(CODE_BUNDLE_ZIP)),
    "--expected-code-bundle-sha256", EXPECTED_CODE_BUNDLE_SHA256,
    "--code-root", str(CODE_ROOT),
    "--data-bundle", str(Path(DATA_BUNDLE_ZIP)),
    "--expected-data-bundle-sha256", EXPECTED_DATA_BUNDLE_SHA256,
    "--runtime-root", str(RUNTIME_ROOT),
    "--durable-dir", str(DURABLE_DIR),
    "--config-template", str(CODE_ROOT / CONFIG_RELATIVE),
    "--runtime-config", str(runtime_config_path),
    "--execution-profile", EXECUTION_PROFILE,
    "--json-out", str(preflight_receipt_path),
]
if REQUIRE_CUDA:
    preflight_command.append("--require-cuda")
if ALLOW_CPU_SMOKE:
    preflight_command.append("--allow-cpu-smoke")
result = subprocess.run(preflight_command)
if result.returncode != 0:
    raise RuntimeError(f"preflight refused the run (exit {result.returncode}); no fit was started")
preflight_receipt = json.loads(preflight_receipt_path.read_text(encoding="utf-8"))
print({k: preflight_receipt[k] for k in ("status", "execution_profile", "device", "estimates")})


In [ ]:
# The one source-of-truth trainer CLI; this notebook contains no training logic.
if RUN_DIR_RUNTIME.exists():
    shutil.rmtree(RUN_DIR_RUNTIME)
RUN_DIR_RUNTIME.mkdir(parents=True, exist_ok=True)
train_command = [
    sys.executable,
    "-m", "ml.camera_coach.train",
    "--config", str(runtime_config_path),
    "--run-dir", str(RUN_DIR_RUNTIME),
]
result = subprocess.run(train_command, cwd=str(CODE_ROOT))
if result.returncode != 0:
    raise RuntimeError(f"trainer rejected or failed the run (exit {result.returncode}); checkpoints remain durable")


In [ ]:
# Atomically publish the runtime run directory to durable storage, then emit the
# result manifest the coordinator continues from.
receipt_path = RUN_DIR_RUNTIME / "receipt.json"
if not receipt_path.is_file():
    raise RuntimeError("trainer did not write receipt.json; nothing to publish")
if RUN_DIR_DURABLE.exists():
    shutil.rmtree(RUN_DIR_DURABLE)
publish_stage = Path(tempfile.mkdtemp(prefix=".publish-", dir=str(RUN_DIR_DURABLE.parent)))
publish_target = publish_stage / RUN_ID
shutil.copytree(RUN_DIR_RUNTIME, publish_target)
os.replace(publish_target, RUN_DIR_DURABLE)
shutil.rmtree(publish_stage, ignore_errors=True)

receipt = json.loads((RUN_DIR_DURABLE / "receipt.json").read_text(encoding="utf-8"))
selected = min(receipt["seeds"], key=lambda item: (item["selected_validation_loss"], item["seed"]))
result_manifest = {
    "schema_id": "camera-coach-colab-result-manifest-v1",
    "run_id": RUN_ID,
    "status": receipt["status"],
    "execution_profile": preflight_receipt["execution_profile"],
    "data_admission": receipt["data_admission"],
    "config_sha256": receipt["config_sha256"],
    "resume_semantic_sha256": receipt["hashes"]["resume_semantic_sha256"],
    "bundle": preflight_receipt["bundle"],
    "device": receipt["environment"]["runtime"],
    "actual_device": selected["device"],
    "peak_vram_bytes": selected["peak_vram_bytes"],
    "selected_seed": receipt["selected_seed"],
    "selected_validation_loss": receipt["selected_validation_loss"],
    "epochs": [
        {
            "epoch": row["epoch"],
            "epoch_seconds": row["epoch_seconds"],
            "seconds_per_optimizer_step": row["seconds_per_optimizer_step"],
            "optimizer_steps": row["optimizer_steps"],
        }
        for row in selected["history"]
    ],
    "resume_lineage": selected["resume_lineage"],
    "checkpoint": selected["best_checkpoint_path"],
    "not_a_quality_claim": True,
    "note": (
        "non_admitted_research dry run: this proves the CLI/preflight/device/resume path, not model quality."
        if receipt["data_admission"]["declared"] != "declared_admitted"
        else "declared admitted: admission is a producer claim; quality gates remain separate."
    ),
}
manifest_path = DURABLE_DIR / "result-manifest.json"
manifest_path.write_text(json.dumps(result_manifest, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps({k: result_manifest[k] for k in ("run_id", "status", "actual_device", "peak_vram_bytes", "selected_seed", "resume_lineage")}, indent=2))
from google.colab import files as colab_files
colab_files.download(str(manifest_path))


## What the coordinator receives

`result-manifest.json` (downloaded above and stored in the durable directory) plus the full durable run directory
with `receipt.json`, per-seed `checkpoint.pt`/`best.pt`, `config.json` and `result.json`. The manifest is the result
the coordinator continues verification from; it is not a model-quality verdict.

## Resume contract

Re-running Run all with the same `RUN_ID` resumes the same run when the checkpoint's `resume_semantic_sha256`
(code/config/data/device) matches. If it differs, preflight refuses and the run must start fresh with an explicit
warm-start; silently resuming a changed contract is forbidden.
